# M1 Notebook 18 — Hypothesis Testing and Effect Size

**Notebook ID:** M1_N18  
**Status:** Runnable first edition  
**Random seed:** 42

> Hypothesis tests quantify evidence against a null model. Effect sizes quantify the magnitude of the observed difference.


## 1. Learning objectives

1. State null and alternative hypotheses.
2. Interpret test statistics, p-values, and significance levels.
3. Perform one-sample, independent-sample, paired, and proportion tests.
4. Distinguish statistical significance from practical importance.
5. Compute Cohen's \(d\), Hedges' \(g\), risk difference, and relative risk.
6. Analyze Type I error, Type II error, power, and sample size.
7. Address multiple testing and responsible decision interpretation.


In [ ]:
from srai_math.utils import environment_info, set_seed
from srai_math.statistics import (
    bonferroni_alpha,
    cohens_d_independent,
    cohens_d_one_sample,
    hedges_g,
    independent_t_test,
    one_sample_mean_power,
    one_sample_proportion_z_test,
    one_sample_t_test,
    paired_t_test,
    relative_risk,
    required_sample_size_one_sample_mean,
    risk_difference,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
set_seed(42)
environment_info()


## 2. Hypotheses

A null hypothesis \(H_0\) specifies a reference model.

An alternative hypothesis \(H_1\) specifies a competing claim.

Example:

\[
H_0:\mu=50,
\qquad
H_1:\mu\neq50.
\]


## 3. Test statistic and p-value

A test statistic measures how far the data depart from the null model.

The p-value is the probability, under \(H_0\), of obtaining a result at least as extreme as the observed result.

It is not the probability that \(H_0\) is true.


## 4. One-sample t-test

In [ ]:
sample = np.array([52, 55, 51, 54, 56, 53, 57, 50], dtype=float)

one_sample_result = one_sample_t_test(
    sample,
    null_mean=50.0,
    alternative="two-sided",
)
one_sample_result


In [ ]:
one_sample_effect = cohens_d_one_sample(
    sample,
    reference=50.0,
)

{
    "p_value": one_sample_result["p_value"],
    "Cohens_d": one_sample_effect,
}


The p-value addresses evidence against the null model. Cohen's \(d\) expresses the mean difference in standard-deviation units.


## 5. Independent-samples comparison

In [ ]:
rng = np.random.default_rng(42)

programme = rng.normal(72, 8, 80)
control = rng.normal(68, 8, 75)

independent_result = independent_t_test(
    programme,
    control,
    equal_variance=False,
)
d_independent = cohens_d_independent(
    programme,
    control,
    pooled=True,
)
g_independent = hedges_g(
    programme,
    control,
)

{
    "mean_programme": programme.mean(),
    "mean_control": control.mean(),
    "p_value": independent_result["p_value"],
    "Cohens_d": d_independent,
    "Hedges_g": g_independent,
}


Hedges' \(g\) applies a small-sample bias correction to the standardized mean difference.


## 6. Paired-samples test

In [ ]:
before = np.array([18, 20, 17, 22, 19, 24, 21, 23], dtype=float)
after = np.array([15, 18, 16, 19, 17, 21, 18, 20], dtype=float)

paired_result = paired_t_test(
    before,
    after,
    alternative="greater",
)

differences = before - after

{
    "mean_difference": differences.mean(),
    "p_value": paired_result["p_value"],
    "paired_effect_size": cohens_d_one_sample(
        differences,
        reference=0.0,
    ),
}


Paired analysis uses within-unit differences and is appropriate when observations are naturally matched.


## 7. One-sample proportion test

In [ ]:
proportion_result = one_sample_proportion_z_test(
    successes=62,
    trials=100,
    null_proportion=0.50,
    alternative="two-sided",
)
proportion_result


## 8. Binary-outcome effect sizes

In [ ]:
rd = risk_difference(
    successes_treatment=30,
    total_treatment=100,
    successes_control=20,
    total_control=100,
)
rr = relative_risk(
    successes_treatment=30,
    total_treatment=100,
    successes_control=20,
    total_control=100,
)

{
    "risk_difference": rd,
    "relative_risk": rr,
}


Risk difference gives the absolute percentage-point change. Relative risk gives the multiplicative change. Both are useful because they answer different questions.


## 9. Type I and Type II errors

- Type I error: reject a true null hypothesis.
- Type II error: fail to reject a false null hypothesis.
- Power:

\[
1-\beta.
\]

Power is the probability of detecting a specified effect under a specified alternative.


## 10. Power curve

In [ ]:
sample_sizes = np.arange(10, 201, 5)
effect_sizes = [0.2, 0.5, 0.8]

fig, ax = plt.subplots(figsize=(8, 5))
for effect in effect_sizes:
    powers = [
        one_sample_mean_power(
            effect_size=effect,
            sample_size=int(n),
            alpha=0.05,
        )
        for n in sample_sizes
    ]
    ax.plot(
        sample_sizes,
        powers,
        label=f"Effect size={effect}",
    )

ax.axhline(0.80, linestyle="--")
ax.set_xlabel("Sample size")
ax.set_ylabel("Power")
ax.set_title("Power versus Sample Size")
ax.legend()
plt.show()


## 11. Required sample size

In [ ]:
required_sizes = pd.DataFrame({
    "standardized_effect_size": [0.2, 0.5, 0.8],
    "required_sample_size_for_80_percent_power": [
        required_sample_size_one_sample_mean(
            effect_size=e,
            power=0.80,
            alpha=0.05,
        )
        for e in [0.2, 0.5, 0.8]
    ],
})
required_sizes


Smaller effects require larger samples to detect reliably.


## 12. p-values and sample size

In [ ]:
true_effect = 0.2
sample_sizes_demo = [25, 100, 1000, 10000]
records = []

for n in sample_sizes_demo:
    rng_local = np.random.default_rng(7)
    data = rng_local.normal(
        loc=true_effect,
        scale=1.0,
        size=n,
    )
    result = one_sample_t_test(
        data,
        null_mean=0.0,
    )
    records.append({
        "sample_size": n,
        "estimated_mean": data.mean(),
        "p_value": result["p_value"],
        "Cohens_d": cohens_d_one_sample(data, 0.0),
    })

pd.DataFrame(records)


A tiny effect can become statistically significant with a very large sample. Practical importance must therefore be examined separately.


## 13. Multiple testing

In [ ]:
family_alpha = 0.05
number_of_tests = 10
adjusted_alpha = bonferroni_alpha(
    family_alpha,
    number_of_tests,
)

{
    "familywise_alpha": family_alpha,
    "number_of_tests": number_of_tests,
    "Bonferroni_per_test_alpha": adjusted_alpha,
}


Bonferroni adjustment controls familywise error conservatively. Other methods may offer more power depending on the inferential objective.


## 14. Null-distribution visualization

In [ ]:
df = sample.size - 1
observed_t = one_sample_result["statistic"]
x = np.linspace(-5, 5, 1000)
density = stats.t.pdf(x, df=df)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, density)
extreme = np.abs(x) >= abs(observed_t)
ax.fill_between(x, 0, density, where=extreme, alpha=0.3)
ax.axvline(observed_t, linestyle="--")
ax.axvline(-observed_t, linestyle="--")
ax.set_xlabel("t statistic under H₀")
ax.set_ylabel("Density")
ax.set_title("Two-Sided p-value as Tail Area")
plt.show()


## 15. Statistics interpretation

Hypothesis testing is one component of inference. Strong reporting combines:

- estimates;
- uncertainty intervals;
- effect sizes;
- p-values;
- assumptions;
- design information;
- practical interpretation.


## 16. AI interpretation

Hypothesis tests support:

- A/B experiments;
- model-comparison studies;
- fairness audits;
- drift monitoring;
- feature evaluation;
- benchmark comparisons.

Repeated model experimentation creates multiplicity and selection-bias risks.


## 17. Decision Intelligence case — Evaluating a public programme

Suppose districts implementing a new service-delivery programme are compared with control districts.


In [ ]:
programme_outcome = np.array([
    74, 79, 71, 76, 82, 75, 78, 80, 77, 73,
], dtype=float)
control_outcome = np.array([
    69, 72, 70, 68, 74, 71, 73, 67, 72, 70,
], dtype=float)

programme_test = independent_t_test(
    programme_outcome,
    control_outcome,
    equal_variance=False,
)
programme_effect = cohens_d_independent(
    programme_outcome,
    control_outcome,
)

{
    "mean_difference": (
        programme_outcome.mean()
        - control_outcome.mean()
    ),
    "p_value": programme_test["p_value"],
    "Cohens_d": programme_effect,
}


### Interpretation

The comparison estimates association under the chosen design. It does not establish causal programme impact unless assignment, confounding, interference, missingness, and measurement issues are adequately addressed.


## 18. Engineering notes

- Pre-specify hypotheses and outcomes.
- Report all planned and unplanned analyses.
- Inspect assumptions and data quality.
- Use robust or nonparametric alternatives when appropriate.
- Correct for multiplicity when many claims are tested.
- Power analysis requires a scientifically meaningful effect size.
- Statistical significance is not decision significance.


## 19. Common errors

- Interpreting a p-value as \(P(H_0\mid data)\).
- Treating failure to reject as proof of no effect.
- Reporting significance without effect size.
- Testing many outcomes and highlighting only significant ones.
- Ignoring dependence or pairing.
- Choosing hypotheses after seeing the data.
- Equating association with causal impact.


## 20. Exercises

### Level A
Explain null hypotheses, p-values, Type I error, and power.

### Level B
Derive the one-sample t statistic.

### Level C
Simulate Type I error and power under different sample sizes.

### Capstone
Evaluate a programme using an appropriate test, report an effect size and confidence interval, assess power, address multiplicity, and explain the causal limitations.


## 21. Key insight

Hypothesis tests quantify how incompatible the data are with a null model. Effect sizes quantify magnitude. Responsible inference combines both with uncertainty, study design, assumptions, and substantive judgment.
